<a href="https://colab.research.google.com/github/Jeevith252/ML_BASIC_PROJECT/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
text = "NLP ,, Session is... going on!!!"
clean = re.sub(r'[^a-zA-Z]' , '' ,text)
print(clean)

NLPSessionisgoingon


In [ ]:
text = "The climate is beautiful"
tokens = text.split()
print(tokens)

['The', 'climate', 'is', 'beautiful']


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
text = "The climate is beautiful"
tokens = word_tokenize(text)
filtered = [
    word for word in tokens
    if word.lower() not in stopwords.words("english")
]
print(filtered)

['climate', 'beautiful']


In [ ]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
words = ["played" , "reading"]
for word in words:
  print(stemmer.stem(word))

play
read


In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
print(lemmatizer.lemmatize("running", pos="v"))

[nltk_data] Downloading package wordnet to /root/nltk_data...


run


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
text = ["The climate is beautiful" , "It is raining"]
vector = CountVectorizer()

In [ ]:
x = vector.fit_transform(text)
print(x.toarray())

[[1 1 1 0 0 1]
 [0 0 1 1 1 0]]


In [ ]:
print(vector.get_feature_names_out())

['beautiful' 'climate' 'is' 'it' 'raining' 'the']


In [ ]:
pip install gensim --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 26.4 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec



sentences = ["climate", "is", "beautiful"],["it", "is", "raining"]
model = Word2Vec(sentences,vector_size=20,window=2,min_count=1,workers=4)

print(model.wv["climate"])

[-0.00428278  0.01413282  0.02700714  0.03526328 -0.02851561  0.0092941
  0.03044432 -0.02399025 -0.0155363   0.03398815  0.00815738  0.00094959
  0.01736819  0.00108889  0.04809413  0.02530302 -0.04458695 -0.0352078
  0.00450728  0.03196267]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

text = ["I love this phone","I hate this phone's feature"]

labels = [1, 0]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(text)
model = LogisticRegression()
model.fit(X, labels)

new_text = ["i like this phone"]
new_x = vectorizer.transform(new_text)
predictions = model.predict(new_x)
print(predictions)

[1]


# **IMDB Project**

In [4]:
!pip install -q nltk spacy xgboost
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 39.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import pandas as pd
import re,string,nltk,spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score , classification_report ,confusion_matrix
nltk.download("punkt_tab")
nltk.download("stopwords")
nlp = spacy.load("en_core_web_sm")



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


**Loading the dataset**

In [6]:
df = pd.read_csv("/content/IMDB Dataset.csv")

In [7]:
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [8]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [9]:


df.describe()

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [10]:
df.isnull().sum()

,0
review,0
sentiment,0


In [11]:
df.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
49995,False
49996,False
49997,False
49998,False


In [12]:
stop = stopwords.words("english")

def preprocess(text):
  text = text.lower()

  text = re.sub(r"<.*?>"," ",text)
  text = re.sub(r'http\S+|www|S+' , ' ' ,text)
  text = re.sub(r'[^a-z\s]',' ',text)

  tokens = word_tokenize(text)
  tokens = [t for t in tokens if t not in stop]

  doc = nlp(" ".join(tokens))

  lemmas = [tok.lemma_ for tok in doc]
  return ' '.join(lemmas)

In [13]:
df["clean_review"] = df["review"].apply(preprocess)

df[["review" , "clean_review"]].head()

KeyboardInterrupt: 

In [18]:
df['label'] = df['sentiment'].map({'negative': 0 , "positive" : 1})

X_train , X_test ,y_train , y_test = train_test_split(df["clean_review"] , df["label"] ,test_size = 0.2 , random_state =42 ,stratify = df["label"])

In [19]:
tfidf = TfidfVectorizer(max_features = 5000)
X_train_t = tfidf.fit_transform(X_train)
X_test_t = tfidf.transform(X_test)
print(X_train_t.shape)

(40000, 5000)


In [21]:
lr = LogisticRegression(max_iter = 1000)
lr.fit(X_train_t , y_train)
pred_lr = lr.predict(X_test_t)
print("LR Accuracy",accuracy_score(y_test , pred_lr))
print(confusion_matrix(y_test,pred_lr))
print(classification_report(y_test,pred_lr))


NameError: name 'LogisticRegression' is not defined

In [22]:
xgb = XGBClassifier(n_estimators = 100 ,max_depth = 6,learning_rate = 0.1, eval_metric = "logloss", random_state =42)
xgb.fit(X_train_t,y_train)
xgb_pred = xgb.predict(X_test_t)
print("XGB accuracy :",accuracy_score(y_test,xgb_pred))
print(confusion_matrix(y_test,xgb_pred))
print(classification_report(y_test,xgb_pred))

XGB accuracy : 0.831
[[4005  995]
 [ 695 4305]]
              precision    recall  f1-score   support

           0       0.85      0.80      0.83      5000
           1       0.81      0.86      0.84      5000

    accuracy                           0.83     10000
   macro avg       0.83      0.83      0.83     10000
weighted avg       0.83      0.83      0.83     10000



In [ ]:
results = pd.DataFrame({
    "Model" : ["Logistic Regression" , "XGBoost"],
    "Accuracy" : [accuracy_score(y_test , pred_lr),
                  accuracy_score(y_test,xgb_pred)]
})

print(results)

In [ ]:
def predict_review(review,model):
  clean = preprocess(review)
  vec = tfidf.transform([clean])
  pred = model.predict(vec)[0]

  return "Positive" if pred == 1 else "Negative"

print(predict_review("This movie was aboslutely fantastic." ,lr))
print(predict_review("Worst movie I have ever watched." , xgb))

##Hugging face transformers



In [14]:
!pip -q install transformers datasets accelerate evaluate scikit-learn

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio

Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.1.6
Uninstalling torchvision-0.1.6:
  Successfully uninstalled torchvision-0.1.6
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.1 

In [1]:
!pip install evaluate

In [3]:

import pandas as pd
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np
from sklearn.model_selection import train_test_split

ModuleNotFoundError: Could not import module 'DistilBertForSequenceClassification'. Are this object's requirements defined correctly?

In [ ]:
import datasets
print(datasets.__version__)

In [ ]:
!pip uninstall -y datasets
!pip install datasets == 3.1.0

In [ ]:
df=pd.read_csv("/content/IMDB Dataset.csv")
df['label']=df['sentiment'].map({'negative':0,'positive':1})
df=df[['review','label']]
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42,stratify=df['label'])
train_ds=Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds=Dataset.from_pandas(test_df.reset_index(drop=True))
train_ds,test_ds

In [ ]:
tokenizer=DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
def tokenize(batch):
  return tokenizer(batch['review'],padding='max_length',truncation=True,max_length=256)

train_ds=train_ds.map(tokenize,batched=True)
test_ds=test_ds.map(tokenize,batched=True)
print("\nColumns after tokenization:")
print(train_ds.column_names)
print("\nFirst tokenized examples:")
print(train_ds[0])
cols=["input_ids","attention_mask","label"]
train_ds.set_format(type='torch',columns=cols)
test_ds.set_format(type='torch',columns=cols)

In [ ]:


model=DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased',num_labels=2)

In [ ]:
accuracy=evaluate.load('accuracy')
precision=evaluate.load('precision')
recall=evaluate.load('recall')
f1=evaluate.load('f1')

def compute_metrics(eval_pred):
  logits , labels = eval_pred
  preds=np.argmax(logits,axis=-1)
  return {
      'accuracy':accuracy.compute(predictions=preds,references=labels)['accuracy'],
      'precision':precision.compute(predictions=preds,references=labels)['precision'],
      'recall':recall.compute(predictions=preds,references=labels)['recall'],
      'f1':f1.compute(predictions=preds,references=labels)['f1']
  }

In [ ]:

training_args=TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100
)

In [ ]:

trainer=Trainer(
    model = model,
    args = training_args,
    train_dataset = train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
!pip install torchvision==0.1.6

In [ ]:
print(trainer.train())
print(trainer.evaluate())

In [ ]:
from transformers import pipeline

classifier = pipeline('sentiment-analysis', model = 'distilbert_imdb', tokenizer = 'distilbert_imdb')

print(classifier('This movie was absolutely fantastic.'))
print(classifier("Worst movie ever made."))